In [ ]:
from collections.abc import Callable, Iterable
from functools import partial
from typing import Literal, NamedTuple

import numpy as np
import scipy.sparse
import tqdm.notebook

from gacha_model import COND_PROB_5_STAR, COND_PROB_6_STAR
from plot_tools import draw_multi_pmf_cdf_fig, draw_pmf_cdf_fig
from random_variable import FiniteDist


In [ ]:
class 状态类(NamedTuple):
    六星水位: int
    五星水位: int
    定向选调水位: int
    已触发定向选调: bool
    已获取非当期UP五星干员: bool
    已获取UP六星干员数量: int
    已获取当期UP五星干员数量: int


def 获取状态(*, 六星水位: int, 五星水位: int, 定向选调水位: int, 已触发定向选调: bool, 已获取非当期UP五星干员: bool, 已获取UP六星干员数量: int, 已获取当期UP五星干员数量: int) -> 状态类:
    return 状态类(
        六星水位=六星水位,
        五星水位=五星水位,
        定向选调水位=min(定向选调水位, 150) if not 已触发定向选调 else 0,
        已触发定向选调=已触发定向选调,
        已获取非当期UP五星干员=已获取非当期UP五星干员,
        已获取UP六星干员数量=min(已获取UP六星干员数量, 已获取UP六星干员数量上限),
        已获取当期UP五星干员数量=min(已获取当期UP五星干员数量, 已获取当期UP五星干员数量上限),
    )


def 状态转移(起始状态: 状态类, *, 抽数范围: Literal["1-9,11-51", "10", "52-101", "102-"]) -> list[tuple[状态类, float]]:
    转移概率列表: list[tuple[状态类, float]] = []

    起始六星水位 = 起始状态.六星水位
    起始五星水位 = 起始状态.五星水位
    起始定向选调水位 = 起始状态.定向选调水位
    起始已触发定向选调 = 起始状态.已触发定向选调
    起始已获取非当期UP五星干员 = 起始状态.已获取非当期UP五星干员
    起始已获取UP六星干员数量 = 起始状态.已获取UP六星干员数量
    起始已获取当期UP五星干员数量 = 起始状态.已获取当期UP五星干员数量

    # 计算抽到不同星级干员的概率
    六星概率 = COND_PROB_6_STAR[起始六星水位]
    if 抽数范围 == "10" and 起始五星水位 == 9:
        五星概率 = 1 - 六星概率
    else:
        五星概率 = np.clip(COND_PROB_5_STAR[起始五星水位], 0, 1 - 六星概率)
    四星或三星概率 = 1 - 六星概率 - 五星概率

    if not 起始已触发定向选调 and 起始定向选调水位 >= 150:
        UP六星占六星 = 1
        已触发定向选调 = True
    else:
        UP六星占六星 = 1 / 2
        已触发定向选调 = 起始已触发定向选调

    if 抽数范围 in ("52-101", "102-") and 起始已获取当期UP五星干员数量 == 0 and not 起始已获取非当期UP五星干员:
        当期UP五星占五星 = 1 / 2
        非当期UP五星占五星 = 1 / 2
    elif 抽数范围 == "102-" and 起始已获取当期UP五星干员数量 >= 1 and not 起始已获取非当期UP五星干员:
        当期UP五星占五星 = 0
        非当期UP五星占五星 = 1
    elif 抽数范围 == "102-" and 起始已获取当期UP五星干员数量 == 0 and 起始已获取非当期UP五星干员:
        当期UP五星占五星 = 1
        非当期UP五星占五星 = 0
    else:
        当期UP五星占五星 = 1 / 4
        非当期UP五星占五星 = 1 / 4

    # 抽到 UP 6★ 干员
    转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 定向选调水位=0, 已触发定向选调=已触发定向选调, 已获取非当期UP五星干员=起始已获取非当期UP五星干员, 已获取UP六星干员数量=起始已获取UP六星干员数量 + 1, 已获取当期UP五星干员数量=起始已获取当期UP五星干员数量), 六星概率 * UP六星占六星))

    # 抽到其他 6★ 干员
    转移概率列表.append((获取状态(六星水位=0, 五星水位=0, 定向选调水位=起始定向选调水位 + 1, 已触发定向选调=起始已触发定向选调, 已获取非当期UP五星干员=起始已获取非当期UP五星干员, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取当期UP五星干员数量=起始已获取当期UP五星干员数量), 六星概率 * (1 - UP六星占六星)))

    # 抽到当期 UP 5★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 定向选调水位=起始定向选调水位 + 1, 已获取非当期UP五星干员=起始已获取非当期UP五星干员, 已触发定向选调=起始已触发定向选调, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取当期UP五星干员数量=起始已获取当期UP五星干员数量 + 1), 五星概率 * 当期UP五星占五星))

    # 抽到非当期 UP 5★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 定向选调水位=起始定向选调水位 + 1, 已获取非当期UP五星干员=True, 已触发定向选调=起始已触发定向选调, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取当期UP五星干员数量=起始已获取当期UP五星干员数量), 五星概率 * 非当期UP五星占五星))

    # 抽到其他 5★ 干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=0, 定向选调水位=起始定向选调水位 + 1, 已获取非当期UP五星干员=起始已获取非当期UP五星干员, 已触发定向选调=起始已触发定向选调, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取当期UP五星干员数量=起始已获取当期UP五星干员数量), 五星概率 * (1 - 当期UP五星占五星 - 非当期UP五星占五星)))

    # 抽到 4★ 及以下干员
    转移概率列表.append((获取状态(六星水位=起始六星水位 + 1, 五星水位=起始五星水位 + 1, 定向选调水位=起始定向选调水位 + 1, 已获取非当期UP五星干员=起始已获取非当期UP五星干员, 已触发定向选调=起始已触发定向选调, 已获取UP六星干员数量=起始已获取UP六星干员数量, 已获取当期UP五星干员数量=起始已获取当期UP五星干员数量), 四星或三星概率))

    转移概率列表 = [(目标状态, 概率) for 目标状态, 概率 in 转移概率列表 if 概率 > 0]

    assert np.isclose(sum(x[1] for x in 转移概率列表), 1)
    return 转移概率列表


def 构造状态转移矩阵(状态转移: Callable[[状态类], Iterable[tuple[状态类, float]]]) -> scipy.sparse.csr_array:
    状态数量 = len(状态列表)
    row = []
    col = []
    data = []
    for 起始状态序号, 起始状态 in tqdm.notebook.tqdm(list(enumerate(状态列表))):
        转移概率列表 = 状态转移(起始状态)
        for 目标状态, 概率 in 转移概率列表:
            目标状态序号 = 状态索引[目标状态]
            row.append(起始状态序号)
            col.append(目标状态序号)
            data.append(概率)
    return scipy.sparse.csr_array((data, (row, col)), shape=(状态数量, 状态数量))


已获取UP六星干员数量上限 = 6
已获取当期UP五星干员数量上限 = 6


状态列表: list[状态类] = []
状态列表.extend(状态类(六星水位=六星水位, 五星水位=五星水位, 定向选调水位=定向选调水位, 已触发定向选调=已触发定向选调, 已获取非当期UP五星干员=已获取非当期UP五星干员, 已获取UP六星干员数量=已获取UP六星干员数量, 已获取当期UP五星干员数量=已获取当期UP五星干员数量)
            for 六星水位 in range(0, 99)
            for 五星水位 in range(0, 40)
            for 定向选调水位 in range(0, 151)
            for 已触发定向选调 in (False, )
            for 已获取非当期UP五星干员 in (False, True)
            for 已获取UP六星干员数量 in range(0, 已获取UP六星干员数量上限 + 1)
            for 已获取当期UP五星干员数量 in range(0, 已获取当期UP五星干员数量上限 + 1)
            )
状态列表.extend(状态类(六星水位=六星水位, 五星水位=五星水位, 定向选调水位=定向选调水位, 已触发定向选调=已触发定向选调, 已获取非当期UP五星干员=已获取非当期UP五星干员, 已获取UP六星干员数量=已获取UP六星干员数量, 已获取当期UP五星干员数量=已获取当期UP五星干员数量)
            for 六星水位 in range(0, 99)
            for 五星水位 in range(0, 40)
            for 定向选调水位 in (0,)
            for 已触发定向选调 in (True, )
            for 已获取非当期UP五星干员 in (False, True)
            for 已获取UP六星干员数量 in range(0, 已获取UP六星干员数量上限 + 1)
            for 已获取当期UP五星干员数量 in range(0, 已获取当期UP五星干员数量上限 + 1)
            )
状态数量: int = len(状态列表)
状态索引: dict[状态类, int] = {状态: i for i, 状态 in enumerate(状态列表)}

In [ ]:
状态转移矩阵_第10抽 = 构造状态转移矩阵(partial(状态转移, 抽数范围="10"))
状态转移矩阵_前51抽且非第10抽 = 构造状态转移矩阵(partial(状态转移, 抽数范围="1-9,11-51"))
状态转移矩阵_第52抽到第101抽 = 构造状态转移矩阵(partial(状态转移, 抽数范围="52-101"))
状态转移矩阵_第102抽及以后 = 构造状态转移矩阵(partial(状态转移, 抽数范围="102-"))

scipy.sparse.save_npz("限时标准寻访中间结果/状态转移矩阵_第10抽.npz", 状态转移矩阵_第10抽)
scipy.sparse.save_npz("限时标准寻访中间结果/状态转移矩阵_前51抽且非第10抽.npz", 状态转移矩阵_前51抽且非第10抽)
scipy.sparse.save_npz("限时标准寻访中间结果/状态转移矩阵_第52抽到第101抽.npz", 状态转移矩阵_第52抽到第101抽)
scipy.sparse.save_npz("限时标准寻访中间结果/状态转移矩阵_第102抽及以后.npz", 状态转移矩阵_第102抽及以后)

In [ ]:
迭代次数 = 4096

初始状态 = 状态类(六星水位=0, 五星水位=0, 定向选调水位=0, 已触发定向选调=False, 已获取非当期UP五星干员=False, 已获取UP六星干员数量=0, 已获取当期UP五星干员数量=0)
当前状态分布 = np.zeros(状态数量)
当前状态分布[状态索引[初始状态]] = 1

历史获得甲乙数量联合分布 = np.zeros((迭代次数 + 1, 已获取UP六星干员数量上限 + 1, 已获取当期UP五星干员数量上限 + 1))
"""`历史获得甲乙数量联合分布[n, i, j]` 表示 n 次寻访后，获取的 (甲, 乙) 干员数量恰好为 (i, j) 的概率"""

状态已获取UP六星干员数量数组 = np.fromiter((s.已获取UP六星干员数量 for s in 状态列表), dtype=int)
状态已获取当期UP五星干员数量数组 = np.fromiter((s.已获取当期UP五星干员数量 for s in 状态列表), dtype=int)

for i in tqdm.notebook.tqdm(range(迭代次数)):
    if i == 9:
        状态转移矩阵 = 状态转移矩阵_第10抽
    elif i < 51:
        状态转移矩阵 = 状态转移矩阵_前51抽且非第10抽
    elif i < 101:
        状态转移矩阵 = 状态转移矩阵_第52抽到第101抽
    else:
        状态转移矩阵 = 状态转移矩阵_第102抽及以后
    当前状态分布 = 当前状态分布 @ 状态转移矩阵

    # 历史状态分布[i + 1] = 当前状态分布

    当前获得甲乙数量联合分布 = np.zeros((已获取UP六星干员数量上限 + 1, 已获取当期UP五星干员数量上限 + 1))
    # for a in range(已获取UP六星干员数量上限 + 1):
    #     for b in range(已获取UP五星干员乙数量上限 + 1):
    #         for c in range(已获取UP五星干员丙数量上限 + 1):
    #             当前获得甲乙丙数量联合分布[a, b, c] = np.sum(当前状态分布[获得特定数量的状态序号列表数组[a, b, c]])
    np.add.at(当前获得甲乙数量联合分布, (状态已获取UP六星干员数量数组, 状态已获取当期UP五星干员数量数组), 当前状态分布)
    # print(当前获得甲乙丙数量联合分布)
    历史获得甲乙数量联合分布[i + 1] = 当前获得甲乙数量联合分布

np.savez_compressed("限时标准寻访中间结果/历史获得甲乙数量联合分布.npz", 历史获得甲乙数量联合分布=历史获得甲乙数量联合分布)

In [ ]:
# 在限时标准寻访中获得若干名 UP 6★ 干员所需寻访次数的分布

历史获得甲乙数量联合分布 = np.load("限时标准寻访中间结果/历史获得甲乙数量联合分布.npz")["历史获得甲乙数量联合分布"]

dists = [FiniteDist([1])]

for n in range(1, 已获取UP六星干员数量上限 + 1):
    已获取UP六星干员数量矩阵 = np.arange(已获取UP六星干员数量上限 + 1)[None, :, None]
    已获取当期UP五星干员数量矩阵 = np.arange(已获取当期UP五星干员数量上限 + 1)[None, None, :]
    mask = 已获取UP六星干员数量矩阵 >= n
    cdf = np.sum(历史获得甲乙数量联合分布 * mask, axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

dists[1] = FiniteDist(dists[1].pk[:249 + 1])

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"{n} 名" for n in range(1, 已获取UP六星干员数量上限 + 1)],
                       title="在限时标准寻访中获得若干名 UP 6★ 干员所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在限时标准寻访中获得 1 名 UP 6★ 干员所需寻访次数的分布",
                 x_max=None,
                 quantile_poses=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99, 1.00],
                 save=True)

In [ ]:
# 在限时标准寻访中获得若干名当期 UP 5★ 干员所需寻访次数的分布

历史获得甲乙数量联合分布 = np.load("限时标准寻访中间结果/历史获得甲乙数量联合分布.npz")["历史获得甲乙数量联合分布"]

dists = [FiniteDist([1])]

for n in range(1, 已获取当期UP五星干员数量上限 + 1):
    已获取UP六星干员数量矩阵 = np.arange(已获取UP六星干员数量上限 + 1)[None, :, None]
    已获取当期UP五星干员数量矩阵 = np.arange(已获取当期UP五星干员数量上限 + 1)[None, None, :]
    mask = 已获取当期UP五星干员数量矩阵 >= n
    cdf = np.sum(历史获得甲乙数量联合分布 * mask, axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"{n} 名" for n in range(1, 已获取当期UP五星干员数量上限 + 1)],
                       title="在限时标准寻访中获得若干名当期 UP 5★ 干员所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在限时标准寻访中获得 1 名当期 UP 5★ 干员所需寻访次数的分布",
                 x_max="auto",
                 save=True)

In [ ]:
# 在限时标准寻访中获得 UP 6★ 干员和当期 UP 5★ 干员各至少若干次所需寻访次数的分布

历史获得甲乙数量联合分布 = np.load("限时标准寻访中间结果/历史获得甲乙数量联合分布.npz")["历史获得甲乙数量联合分布"]

dists = [FiniteDist([1])]

数量上限 = min(已获取UP六星干员数量上限, 已获取当期UP五星干员数量上限)

for n in range(1, 数量上限 + 1):
    已获取UP六星干员数量矩阵 = np.arange(已获取UP六星干员数量上限 + 1)[None, :, None]
    已获取当期UP五星干员数量矩阵 = np.arange(已获取当期UP五星干员数量上限 + 1)[None, None, :]
    mask = (已获取UP六星干员数量矩阵 >= n) & (已获取当期UP五星干员数量矩阵 >= n)
    cdf = np.sum(历史获得甲乙数量联合分布 * mask, axis=(1, 2))
    dist = FiniteDist(cdf=cdf)
    dists.append(dist)

draw_multi_pmf_cdf_fig(dists=dists[1:],
                       labels=[f"各至少 {n} 次" for n in range(1, 数量上限 + 1)],
                       title="在限时标准寻访中获得 UP 6★ 干员和当期 UP 5★ 干员各至少若干次所需寻访次数的分布",
                       x_max="auto",
                       save=True)

draw_pmf_cdf_fig(dist=dists[1],
                 title="在限时标准寻访中获得 UP 6★ 干员和当期 UP 5★ 干员各至少 1 次所需寻访次数的分布",
                 x_max="auto",
                 save=True)